In [3]:
from datetime import datetime, timedelta
import time
import numpy as np
import pandas as pd
import requests

CITY_NAME = "Faisalabad"
LATITUDE = 31.4187
LONGITUDE = 73.0791
PARQUET_FILE_PATH = "faisalabad_aqi_2years.parquet"


def fetch_weather_and_air_quality_chunk(
    start_date: str, end_date: str
) -> pd.DataFrame:
    """Fetches weather and air quality for a specific date range."""
    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "surface_pressure",
            "wind_speed_10m",
            "wind_direction_10m",
        ],
        "timezone": "UTC",
    }

    aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    aq_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["pm2_5", "pm10", "nitrogen_dioxide", "ozone", "us_aqi"],
        "timezone": "UTC",
    }

    print(f"Fetching data from {start_date} to {end_date}...")
    res_weather = requests.get(weather_url, params=weather_params).json()
    res_aq = requests.get(aq_url, params=aq_params).json()

    df_weather = pd.DataFrame(res_weather["hourly"])
    df_weather["time"] = pd.to_datetime(df_weather["time"])

    df_aq = pd.DataFrame(res_aq["hourly"])
    df_aq["time"] = pd.to_datetime(df_aq["time"])

    df = pd.merge(df_weather, df_aq, on="time", how="inner")
    df["city"] = CITY_NAME

    return df

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Computes full feature engineering set adhering to Hopsworks schema."""
    df = df.sort_values("time").reset_index(drop=True)

    # 1. Wind Vector Components
    wind_rad = np.radians(df["wind_direction_10m"])
    df["wind_u"] = -df["wind_speed_10m"] * np.sin(wind_rad)
    df["wind_v"] = -df["wind_speed_10m"] * np.cos(wind_rad)

    # 2. Temporal & Cyclical Features
    df["hour"] = df["time"].dt.hour.astype("int64")
    df["dayofweek"] = df["time"].dt.dayofweek.astype("int64")
    df["month"] = df["time"].dt.month.astype("int64")
    df["dayofyear"] = df["time"].dt.dayofyear.astype("int64")
    df["is_weekend"] = (
        df["dayofweek"].apply(lambda x: 1 if x >= 5 else 0).astype("int64")
    )

    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12.0)

    # 3. Lag Features
    df["pm2_5_lag_1h"] = df["pm2_5"].shift(1)
    df["pm2_5_lag_24h"] = df["pm2_5"].shift(24)
    df["us_aqi_lag_1h"] = df["us_aqi"].shift(1)
    df["us_aqi_lag_24h"] = df["us_aqi"].shift(24)
    df["us_aqi_lag_48h"] = df["us_aqi"].shift(48)
    df["us_aqi_lag_72h"] = df["us_aqi"].shift(72)

    # 4. Rolling Features
    df["pm2_5_roll_mean_6h"] = df["pm2_5"].shift(1).rolling(window=6).mean()
    df["pm2_5_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(window=24).mean()
    df["us_aqi_roll_mean_24h"] = (
        df["us_aqi"].shift(1).rolling(window=24).mean()
    )

    # 5. Rates of Change
    df["aqi_change_rate_1h"] = (
        df["pm2_5_lag_1h"] - df["pm2_5"].shift(2)
    ) / (df["pm2_5"].shift(2) + 1e-5)
    df["aqi_change_rate_24h"] = (
        df["pm2_5_lag_1h"] - df["pm2_5_lag_24h"]
    ) / (df["pm2_5_lag_24h"] + 1e-5)

    # 6. Target Horizons
    df["target_aqi_24h"] = df["us_aqi"].shift(-24)
    df["target_aqi_48h"] = df["us_aqi"].shift(-48)
    df["target_aqi_72h"] = df["us_aqi"].shift(-72)

    if "day" in df.columns:
        df = df.drop(columns=["day"])

    # Drop nulls generated by shifting/rolling
    df = df.dropna().reset_index(drop=True)

    # Datetime casting for Hopsworks compatibility
    df["city"] = df["city"].astype(str)
    df["us_aqi"] = df["us_aqi"].astype("int64")
    df["time"] = (
        pd.to_datetime(df["time"])
        .dt.tz_localize("UTC")
        .dt.tz_localize(None)
        .astype("datetime64[us]")
    )

    return df

def main():
    today = datetime.now()
    two_years_ago = today - timedelta(days=730)

    # Define 1-year date chunks
    midpoint = two_years_ago + timedelta(days=365)

    chunks = [
        (
            two_years_ago.strftime("%Y-%m-%d"),
            midpoint.strftime("%Y-%m-%d"),
        ),
        (
            (midpoint + timedelta(days=1)).strftime("%Y-%m-%d"),
            today.strftime("%Y-%m-%d"),
        ),
    ]

    raw_dfs = []
    for start_date, end_date in chunks:
        df_chunk = fetch_weather_and_air_quality_chunk(
            start_date, end_date
        )
        raw_dfs.append(df_chunk)
        time.sleep(1)  # Respect API rate limits

    # Combine 2 years of raw hourly data
    full_raw_df = pd.concat(raw_dfs, ignore_index=True)

    # Apply continuous feature engineering across full timeline
    print("Engineering features across 2-year dataset...")
    features_df = engineer_features(full_raw_df)

    # Export to Parquet
    features_df.to_parquet(PARQUET_FILE_PATH, index=False)
    print(
        f"Successfully created {PARQUET_FILE_PATH} with {len(features_df)} rows and {len(features_df.columns)} columns."
    )


if __name__ == "__main__":
    main()

Fetching data from 2024-08-18 to 2025-08-18...
Fetching data from 2025-08-19 to 2026-08-18...
Engineering features across 2-year dataset...
Successfully created faisalabad_aqi_2years.parquet with 17400 rows and 37 columns.


In [ ]:
import pandas as pd

# 1. Read the parquet file
file_path = "faisalabad_aqi_2years.parquet"
df = pd.read_parquet(file_path)

# 2. Transform 'time' to ISO string format (YYYY-MM-DD HH:MM:SS)
if pd.api.types.is_numeric_dtype(df["time"]):
    # Handles epoch timestamps (milliseconds or seconds)
    unit = "ms" if df["time"].iloc[0] > 1e11 else "s"
    df["time"] = pd.to_datetime(df["time"], unit=unit)

# Convert to ISO string format
df["time"] = pd.to_datetime(df["time"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# 3. Display top 10 rows
print(df.tail(20).to_string(index=False))

               time  temperature_2m  relative_humidity_2m  surface_pressure  wind_speed_10m  wind_direction_10m  pm2_5  pm10  nitrogen_dioxide  ozone  us_aqi       city     wind_u   wind_v  hour  dayofweek  month  dayofyear  is_weekend  sin_hour      cos_hour  sin_month  cos_month  pm2_5_lag_1h  pm2_5_lag_24h  us_aqi_lag_1h  us_aqi_lag_24h  us_aqi_lag_48h  us_aqi_lag_72h  pm2_5_roll_mean_6h  pm2_5_roll_mean_24h  us_aqi_roll_mean_24h  aqi_change_rate_1h  aqi_change_rate_24h  target_aqi_24h  target_aqi_48h  target_aqi_72h
2024-08-21 00:00:00            25.6                    95             982.8            10.5                  94   29.3  49.5               8.8   82.0      98 Faisalabad -10.474423 0.732443     0          2      8        234           0  0.000000  1.000000e+00  -0.866025       -0.5          41.1           33.4           97.0            94.0           113.0            84.0           45.866667            34.279167             93.791667           -0.092715             0.230

: 